In [6]:
# Generate a dataframe that has the dates from the start of 2019 up to 4 full calendar years in to the future from when the code was last run

from pyspark.sql import functions as F

dates_df = (

    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit("2019-01-01")),
                F.date_sub(
                    F.add_months(
                        F.trunc(F.current_date(), "year"), 72), 1
                ),
                F.expr("INTERVAL 1 DAY")
            )
        )
        .alias("date")
    )
)



StatementMeta(, b9c98985-70e0-494c-8b8b-b9941e4804dd, 8, Finished, Available, Finished, False)

In [7]:
# Add a date key column in the same format as it is in the rest of the project yyyyMM

dates_df = dates_df.withColumn("year_month", F.date_format(F.col("date"), "yyyyMM").cast("int"))

StatementMeta(, b9c98985-70e0-494c-8b8b-b9941e4804dd, 9, Finished, Available, Finished, False)

In [8]:
# Add year, month number & name and d

dates_df = dates_df.withColumns({
    "year": F.year("date"),
    "quarter": F.quarter("date"),
    "month_num": F.month("date"),
    "month_MON": F.date_format("date", "MMM"),
    "day_of_month": F.dayofmonth("date"),
    "week_of_year": F.weekofyear("date"),
    "dayname": F.date_format("date", "EEEE"),
    "day_of_week": F.pmod(F.dayofweek("date") + 5, 7) + 1
})

StatementMeta(, b9c98985-70e0-494c-8b8b-b9941e4804dd, 10, Finished, Available, Finished, False)

In [9]:
display(dates_df)

StatementMeta(, b9c98985-70e0-494c-8b8b-b9941e4804dd, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0d52400e-d7b2-4f1c-a468-99e543229334)

In [10]:
(
    dates_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_date")
)

StatementMeta(, b9c98985-70e0-494c-8b8b-b9941e4804dd, 12, Finished, Available, Finished, False)